# TrueGauge — fuel-side simulation prototype

Companion to Section V-C of the paper (*Simulation-Based Pre-Validation*).

**What this is.** A small closed-loop simulator of a port-injected spark-ignition engine under lambda control. It shows how a fuel change or a non-fuel fault moves short-term and long-term fuel trims, so the study design can be tested before real vehicle data exist.

**What this is not.** Every property value is an assumed textbook value and the ECU is idealised. Nothing here is measured vehicle data, so it must **never** be used to claim accuracy, sensitivity, or false-positive rates. Simulated and real records are not to be pooled. Diesel engines and the EV charger/battery model are specified in the paper but not implemented here.

**How to run:** Colab -> Runtime -> Run all. No GPU needed.

## 0. Setup

In [ ]:
!pip install -q numpy matplotlib scipy

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq
print('ready')

## 1. Simulator core
Mixture physics, drive cycles, and the ECU/plant loop. Property values are assumptions to be replaced by measured ones.

In [ ]:
# --- approximate property table (nominal textbook values; to be replaced by measured values) -------------
RHO = dict(gasoline=0.745, ethanol=0.789, methanol=0.792, acetone=0.784, kerosene=0.800, water=1.000)   # kg/L
AFR = dict(gasoline=14.7, ethanol=9.0, methanol=6.47, acetone=9.5, kerosene=14.7, water=0.0)            # stoich. air/fuel, mass


def mixture(vol_fracs):
    """vol_fracs: dict of volume fractions summing to 1 -> (density kg/L, stoichiometric air per kg mixture)."""
    tot = sum(vol_fracs.values())
    assert abs(tot - 1.0) < 1e-9, tot
    rho = sum(f * RHO[k] for k, f in vol_fracs.items())
    w = {k: f * RHO[k] / rho for k, f in vol_fracs.items()}            # mass fractions
    afr = sum(w[k] * AFR[k] for k in w)                                  # air needed adds linearly by mass
    return rho, afr


def steady_trim(adulterant, phi, nominal="gasoline"):
    """Idealised steady-state total fuel trim (fraction) needed to hold lambda = 1 when the ECU assumes clean
    gasoline: (1 + trim) = (AFR_nom / AFR_mix) * (rho_nom / rho_mix).  Injectors deliver VOLUME, so density matters."""
    rho_n, afr_n = mixture({nominal: 1.0})
    rho_m, afr_m = mixture({nominal: 1.0 - phi, adulterant: phi})
    return (afr_n / afr_m) * (rho_n / rho_m) - 1.0


def density_temp_factor(dT, beta=0.001):
    """Relative change of fuel density for a temperature change dT (deg C); ~0.1 %/deg C for hydrocarbons."""
    return -beta * dT


# --- drive cycles ----------------------------------------------------------------------------------------
def drive_cycle(kind, n, rng):
    """Intake air mass flow (g/s) per 1-s step."""
    m = np.zeros(n)
    if kind == "highway":
        x = 14.0
        for i in range(n):
            x = 0.97 * x + 0.03 * 14.0 + rng.normal(0, 0.45)
            m[i] = max(x, 6.0)
        return m
    # urban stop-go: idle segments and acceleration/cruise bursts
    i = 0
    while i < n:
        idle = int(rng.integers(8, 30)); seg = np.full(idle, rng.uniform(2.2, 3.2))
        burst = int(rng.integers(10, 40)); lvl = rng.uniform(6.0, 24.0)
        ramp = np.linspace(seg[-1], lvl, burst) + rng.normal(0, 0.4, burst)
        for part in (seg, np.clip(ramp, 2.0, None)):
            k = min(len(part), n - i); m[i:i + k] = part[:k]; i += k
            if i >= n: break
    return m


# --- ECU + plant -----------------------------------------------------------------------------------------
def simulate(vol_fracs=None, cycle="urban", T=2700, seed=0, ltft0=0.0, maf_gain=0.0, inj_gain=0.0,
             leak_gps=0.0, warmup=90, Ks=0.4, KL=1.0 / 900.0, lam_noise=0.008, lam_tau=1.0, clip=0.25):
    """
    Returns dict of arrays (1 Hz).  Fuel is the mixture in `vol_fracs` (default clean gasoline).
    Faults: maf_gain (relative MAF gain error), inj_gain (relative injector flow error), leak_gps (unmetered air).
    ECU: injected volume = m_air_meas / (AFR_nom * rho_nom) * (1 + LTFT + STFT); STFT is a fast integrator on the
    lambda error, LTFT a slow integrator on STFT (time constant 1/KL ~ 15 min). Open loop for `warmup` seconds.
    """
    rng = np.random.default_rng(seed)
    vol_fracs = vol_fracs or {"gasoline": 1.0}
    rho_mix, afr_mix = mixture(vol_fracs)
    rho_nom, afr_nom = mixture({"gasoline": 1.0})
    m_true = drive_cycle(cycle, T, rng)
    stft = np.zeros(T); ltft = np.zeros(T); lam = np.zeros(T); lam_meas = np.zeros(T)
    S, L, lm = 0.0, ltft0, 1.0
    for t in range(T):
        m_meas = (1.0 + maf_gain) * m_true[t]                      # what the ECU believes
        V = m_meas / (afr_nom * rho_nom) * (1.0 + L + S)           # commanded volume (L per s)
        fuel_mass = rho_mix * V * (1.0 + inj_gain)
        lam_t = (m_true[t] + leak_gps) / (fuel_mass * afr_mix)    # leak air is real but unmetered
        lm += (lam_t - lm) / lam_tau + rng.normal(0, lam_noise)    # lagged, noisy lambda sensor
        if t >= warmup:                                            # closed loop
            S = float(np.clip(S + Ks * (lm - 1.0), -clip, clip))   # lean (lm>1) -> add fuel
            L = float(np.clip(L + KL * S, -clip, clip))
        else:
            S = 0.0
        stft[t], ltft[t], lam[t], lam_meas[t] = S, L, lam_t, lm
    return dict(t=np.arange(T), m_air=m_true, stft=stft, ltft=ltft, total=stft + ltft, lam=lam, lam_meas=lam_meas,
                rho=rho_mix, afr=afr_mix)


def settled_total(ltft0=0.0, **kw):
    """Total trim averaged over the last 10 min of a 45-min run."""
    r = simulate(ltft0=ltft0, **kw)
    return float(np.mean(r["total"][-600:]))


def trim_vs_airflow(r, tmin=1500, bins=(2, 4, 6, 8, 10, 13, 16, 20, 26)):
    t = r["t"] >= tmin
    m, y = r["m_air"][t], r["total"][t]
    idx = np.digitize(m, bins)
    xs, ys, es = [], [], []
    for b in range(1, len(bins)):
        sel = idx == b
        if sel.sum() >= 15:
            xs.append(m[sel].mean()); ys.append(y[sel].mean()); es.append(y[sel].std() / np.sqrt(sel.sum()))
    return np.array(xs), np.array(ys), np.array(es)


## 2. Idealised steady-state trim shift (paper Table VIII)

For a petrol-calibrated ECU, holding lambda = 1 needs a total trim of $(\mathrm{AFR}_0/\mathrm{AFR})\,(\rho_0/\rho) - 1$, because injectors deliver **volume**. Water is limited to 5 % and diesel / used cooking oil are not modelled.

In [ ]:
levels = [0.025, 0.05, 0.10, 0.20]
print(f"{'adulterant':22s}" + "".join(f"{int(l*1000)/10:>8}%" for l in levels))
for label, key in [("Ethanol","ethanol"),("Methanol","methanol"),("Acetone (ketone)","acetone"),("Kerosene / paraffin","kerosene"),("Water","water")]:
    cells = ["      --" if (key=="water" and p>0.05) else f"{100*steady_trim(key,p):+8.1f}" for p in levels]
    print(f"{label:22s}" + "".join(cells))
print()
print("A 30 C warmer fuel changes density by", f"{100*density_temp_factor(30):+.1f}%", "-> trim shift", f"{100*(1/(1+density_temp_factor(30))-1):+.1f}%", "for an ECU without density compensation")

## 3. Does the dynamic simulator agree with the analytic prediction?
A consistency check of the code, not a validation against real vehicles.

In [ ]:
KL = 1/500   # long-term trim learning rate, chosen so settling falls inside the 10-30 min design range
for a, p in [("ethanol",0.20),("methanol",0.10),("kerosene",0.10),("water",0.05)]:
    sim = settled_total(vol_fracs={"gasoline":1-p, a:p}, seed=1, KL=KL)
    print(f"{a:9s} {p:.2f}  simulated {100*sim:+.2f}%   analytic {100*steady_trim(a,p):+.2f}%")
print("MAF gain +4%  ->", f"{100*settled_total(maf_gain=0.04, seed=2, KL=KL):+.2f}%")
print("injector +4%  ->", f"{100*settled_total(inj_gain=0.04, seed=2, KL=KL):+.2f}%   (multiplicative faults mimic a fuel change in steady state)")
r = simulate(vol_fracs={"gasoline":.8,"ethanol":.2}, seed=3, KL=KL)
fin = r["ltft"][-300:].mean()
print("E20: LTFT reaches 90% of its final value after", round(np.argmax(r["ltft"]>=0.9*fin)/60,1), "min; total trim responds within ~2 min")

## 4. Figure: learned trim after a fuel change, and the trim-versus-airflow signature (paper Fig. 10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), gridspec_kw=dict(width_ratios=[1.25, 1]))
ax = axes[0]
cases = [("Clean fuel", {}, "#888780", "-"),
         ("E20 on a non-flex vehicle", dict(vol_fracs={"gasoline":.8,"ethanol":.2}), "#EF9F27", "-"),
         ("Methanol 10%", dict(vol_fracs={"gasoline":.9,"methanol":.1}), "#D85A30", "-"),
         ("Kerosene 10%", dict(vol_fracs={"gasoline":.9,"kerosene":.1}), "#1D9E75", "-"),
         ("Pre-existing MAF gain +4% (no fuel change)", dict(maf_gain=0.04, ltft0=-0.0382), "#7F77DD", "--")]
for label, kw, col, ls in cases:
    r = simulate(seed=3, KL=KL, **kw)
    ax.plot(r["t"]/60, 100*r["ltft"], color=col, lw=1.8, ls=ls, label=label+" (LTFT)")
    if label.startswith("E20"):
        k=45; ax.plot(r["t"][k-1:]/60, 100*np.convolve(r["total"], np.ones(k)/k, mode="valid"), color=col, lw=1, ls=":", label="E20 total trim (STFT + LTFT)")
ax.axvspan(0,10,color="#FAEEDA",alpha=.75); ax.axvspan(10,30,color="#FAEEDA",alpha=.38); ax.axvline(0,color="k",lw=.8)
ax.set_xlim(0,45); ax.set_ylim(-6,12.5); ax.set_xlabel("Minutes after refuel (simulated)"); ax.set_ylabel("Fuel trim (%)")
ax.set_title("(a) Learned trim after a fuel change"); ax.legend(fontsize=7, loc="upper center", bbox_to_anchor=(0.5,-0.16), ncol=2, frameon=False)
ax = axes[1]
for label, kw, col in [("E20 (fuel change)", dict(vol_fracs={"gasoline":.8,"ethanol":.2}), "#EF9F27"), ("MAF gain +4%", dict(maf_gain=0.04), "#7F77DD"), ("Vacuum leak 0.5 g/s", dict(leak_gps=0.5), "#E24B4A")]:
    x, y, e = trim_vs_airflow(simulate(seed=4, KL=KL, **kw))
    ax.errorbar(x, 100*y, yerr=100*e, color=col, marker="o", ms=3, lw=1.4, capsize=2, label=label)
ax.axhline(0,color="k",lw=.6); ax.set_xlabel("Intake air mass flow (g/s)"); ax.set_ylabel("Settled total trim (%)")
ax.set_title("(b) Trim signature versus airflow"); ax.legend(fontsize=8, frameon=False)
plt.tight_layout(); plt.show()

## 5. Confusability: which adulterant mixes give the same trim shift?

In [ ]:
target = 0.033
for a in ["ethanol","acetone","methanol"]:
    p = brentq(lambda x: steady_trim(a, x) - target, 0.001, 0.5)
    print(f"a +3.3% trim shift is produced by about {100*p:.1f}% {a}")
print("\nKerosene at 10% shifts trims by only", f"{100*steady_trim('kerosene',0.10):+.2f}%", "-> trims alone are unlikely to identify hydrocarbon adulterants;")
print("their detectability (knock retard, other signals, Track 2) is an open empirical question.")

## 6. Reading the results honestly

* **Noise floor.** With the assumed lambda-sensor noise the ten-minute trim estimate has a floor of about 0.04 %. Real detectability will be set by confounders (fuel temperature, MAF/injector gain, vehicle-to-vehicle baselines), not by sensor noise, so no detectability score is reported.
* **Hypotheses for the empirical study.** (1) Compute features relative to the same vehicle's pre-refuel baseline. (2) Use the trim-versus-airflow signature to separate additive air leaks from multiplicative fuel or gain effects.
* **What would replace the assumptions.** Measured fuel properties (Table VII reference methods), measured ECU behaviour on real vehicles, and a small real subset for calibration and sim-to-real gap analysis.